In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

os.chdir("C:/Training/Kaggle/Competitions/Playground_Competitions/Rainfall(S5E3)")

train = pd.read_csv("train.csv", index_col=0)
test = pd.read_csv("test.csv", index_col=0)

train.columns

X, y = train.drop('rainfall', axis=1), train['rainfall']

imp = SimpleImputer(strategy='median')
lr = LogisticRegression()
pipe = Pipeline([('IMP',imp),('LR',lr)])
pipe.fit(X, y)

y_pred_prob = pipe.predict_proba(test)

submit = pd.read_csv("sample_submission.csv")
submit.head(3)

submit['rainfall'] = y_pred_prob[:,1]
submit.to_csv("sbt_lr.csv", index=False)

X.shape

scaler = StandardScaler()
prcomp = PCA()
pipe_pca = Pipeline([('SCL',scaler),('PCA',prcomp)])

pipe_pca.fit(X)

np.cumsum(prcomp.explained_variance_ratio_)

pipe = Pipeline([('SCL',scaler),('PCA',prcomp),('LR', lr)])
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=25)
params = {'PCA__n_components':[2,3,4,5,6,7,8,9,10]}
gcv = GridSearchCV(pipe, param_grid=params, cv=kfold, n_jobs=-1, scoring='roc_auc')
gcv.fit(X, y)

gcv.best_params_, gcv.best_score_

# Trying for some less components

prcomp = PCA(n_components=6)
lr = LogisticRegression(penalty=None)
pipe = Pipeline([('IMP',imp),('SCL',scaler),('PCA',prcomp),('LR', lr)])
pipe.fit(X,y)

y_pred_prob = pipe.predict_proba(test)
submit['rainfall'] = y_pred_prob[:,1]
submit.to_csv("sbt_pca_lr.csv", index=False)



prcomp = PCA(n_components=6)
svm = SVC(kernel='linear', probability=True, random_state=25)
pipe = Pipeline([('IMP',imp),('SCL',scaler),('PCA',prcomp),('SVM', svm)])
pipe.fit(X,y)

y_pred_prob = pipe.predict_proba(test)
submit['rainfall'] = y_pred_prob[:,1]
submit.to_csv("sbt_pca_svm.csv", index=False)